In [11]:
import asyncio
import csv
import importlib
import inspect
import json
import sys
import time
from pathlib import Path


def find_project_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")


PROJECT_DIR = find_project_dir(Path.cwd().resolve())
SRC_DIR = PROJECT_DIR / "src"

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import loader as loader_pkg
import loader.crawl as crawl_pkg
import loader.crawl.detail_page as detail_page_module
import loader.crawl.performance_record as performance_record_module
import scripts.vpn_switcher as vpn_switcher_module

detail_page_module = importlib.reload(detail_page_module)
performance_record_module = importlib.reload(performance_record_module)
crawl_pkg = importlib.reload(crawl_pkg)
loader_pkg = importlib.reload(loader_pkg)
vpn_switcher = importlib.reload(vpn_switcher_module)

BlockedRequestError = loader_pkg.BlockedRequestError
MissingPerformanceRecordError = loader_pkg.MissingPerformanceRecordError
SlowResponseError = loader_pkg.SlowResponseError
StaleListingError = loader_pkg.StaleListingError
create_browser_context = loader_pkg.create_browser_context
crawl_detail_page = loader_pkg.crawl_detail_page
download_performance_record_pdf = loader_pkg.download_performance_record_pdf

print("프로젝트 경로:", PROJECT_DIR)
print("crawl_detail_page 시그니처:", inspect.signature(crawl_detail_page))
print("download_performance_record_pdf 시그니처:", inspect.signature(download_performance_record_pdf))
assert "retry_count" in inspect.signature(download_performance_record_pdf).parameters


프로젝트 경로: /home/wonbeenlee/WorkSpace/boot/SKN28-1st-4team/data_collection
crawl_detail_page 시그니처: (detail_url: str, *, timeout_ms: int = 30000, context=None) -> loader.crawl.detail_page.DetailPage
download_performance_record_pdf 시그니처: (detail_url: str, output_dir: str | pathlib.Path, *, timeout_ms: int = 20000, retry_count: int = 3, retry_backoff_seconds: float = 2.0, raise_on_blocked_error: bool = False, context=None) -> pathlib.Path | None


In [12]:
CSV_PATH = PROJECT_DIR / "raw" / "page_urls" / "search_scope.csv"
OUTPUT_DIR = SRC_DIR / "output" / "performance_record"
STATE_DIR = SRC_DIR / "output" / "_state"
CHECKPOINT_PATH = STATE_DIR / "crawl_checkpoint.json"

MAX_ITEMS = 10
BATCH_SIZE = 3
DETAIL_CONCURRENCY = 3
PDF_CONCURRENCY = 2
LOG_EVERY = 10
DETAIL_TIMEOUT_MS = 20_000
PDF_TIMEOUT_MS = 20_000
PDF_RETRY_COUNT = 1
PDF_RETRY_BACKOFF_SECONDS = 0.0
BLOCKED_ERROR_THRESHOLD = 1
SLOW_RETRY_LIMIT = 2
SLOW_TO_BLOCK_THRESHOLD = 3
BATCH_SLOW_BLOCK_THRESHOLD = 2

VPN_PRIMARY_TARGETS = ["kr115", "kr142", "kr106", "kr117", "kr153", "kr94", "kr120"]
VPN_FALLBACK_TARGETS = ["jp789", "jp515", "jp429", "jp706", "jp765", "jp570", "jp589", "jp584"]
VPN_CONNECT_TIMEOUT_SECONDS = 45.0
VPN_READY_TIMEOUT_SECONDS = 90.0
VPN_STATUS_POLL_INTERVAL_SECONDS = 2.0
VPN_PROBE_URL = "https://www.carku.kr/search/car-detail.html?wDemoNo=0361001191"
VPN_PROBE_TIMEOUT_SECONDS = 15.0
VPN_MAX_PROBE_LATENCY_SECONDS = 6.0
VPN_ROTATION_MAX_ATTEMPTS = len(VPN_PRIMARY_TARGETS) + len(VPN_FALLBACK_TARGETS)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STATE_DIR.mkdir(parents=True, exist_ok=True)

print(f"CSV 경로: {CSV_PATH}")
print(f"PDF 출력 경로: {OUTPUT_DIR}")
print(f"체크포인트 경로: {CHECKPOINT_PATH}")
print(f"상세 페이지 동시 처리 개수: {DETAIL_CONCURRENCY}")
print(f"PDF 동시 처리 개수: {PDF_CONCURRENCY}")
print(f"상세 페이지 타임아웃(ms): {DETAIL_TIMEOUT_MS}")
print(f"PDF 타임아웃(ms): {PDF_TIMEOUT_MS}")
print(f"느린 응답 재시도 횟수: {SLOW_RETRY_LIMIT}")
print(f"개별 URL 느린 응답 차단 승격 기준: {SLOW_TO_BLOCK_THRESHOLD}")
print(f"배치 느린 응답 차단 승격 기준: {BATCH_SLOW_BLOCK_THRESHOLD}")
print(f"VPN 1순위 서버: {VPN_PRIMARY_TARGETS}")
print(f"VPN 2순위 서버: {VPN_FALLBACK_TARGETS}")
print("차단 감지 시: 한국 서버 우선 -> 전부 실패 시 일본 서버 -> 같은 pending rows 재시도")


CSV 경로: /home/wonbeenlee/WorkSpace/boot/SKN28-1st-4team/data_collection/raw/page_urls/search_scope.csv
PDF 출력 경로: /home/wonbeenlee/WorkSpace/boot/SKN28-1st-4team/data_collection/src/output/performance_record
체크포인트 경로: /home/wonbeenlee/WorkSpace/boot/SKN28-1st-4team/data_collection/src/output/_state/crawl_checkpoint.json
상세 페이지 동시 처리 개수: 3
PDF 동시 처리 개수: 2
상세 페이지 타임아웃(ms): 20000
PDF 타임아웃(ms): 20000
느린 응답 재시도 횟수: 2
개별 URL 느린 응답 차단 승격 기준: 3
배치 느린 응답 차단 승격 기준: 2
VPN 1순위 서버: ['kr115', 'kr142', 'kr106', 'kr117', 'kr153', 'kr94', 'kr120']
VPN 2순위 서버: ['jp789', 'jp515', 'jp429', 'jp706', 'jp765', 'jp570', 'jp589', 'jp584']
차단 감지 시: 한국 서버 우선 -> 전부 실패 시 일본 서버 -> 같은 pending rows 재시도


In [13]:
def build_default_runtime_state() -> dict[str, object]:
    return {
        "vpn_target_index": 0,
        "vpn_fallback_target_index": 0,
        "vpn_switch_history": [],
        "last_vpn_result": None,
        "last_public_ip": None,
    }


def normalize_runtime_state(runtime_state: dict[str, object] | None) -> dict[str, object]:
    payload = build_default_runtime_state()
    if runtime_state:
        payload.update(runtime_state)
    payload.setdefault("vpn_switch_history", [])
    payload.setdefault("vpn_target_index", 0)
    payload.setdefault("vpn_fallback_target_index", 0)
    payload.setdefault("last_vpn_result", None)
    payload.setdefault("last_public_ip", None)
    return payload


def load_initial_targets() -> tuple[list[dict[str, str]], list[dict[str, object]], list[dict[str, object]], dict[str, object], bool]:
    if CHECKPOINT_PATH.exists():
        payload = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
        pending_rows = payload.get("pending_rows", [])
        for fallback_index, row in enumerate(pending_rows, start=1):
            row.setdefault("source_index", fallback_index)
        results = payload.get("results", [])
        failed_rows = payload.get("failed_rows", [])
        runtime_state = normalize_runtime_state(payload.get("runtime_state"))
        return pending_rows, results, failed_rows, runtime_state, True

    with CSV_PATH.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        pending_rows = [
            {**row, "source_index": index}
            for index, row in enumerate(reader, start=1)
            if row.get("detail_url")
        ][:MAX_ITEMS]
    return pending_rows, [], [], build_default_runtime_state(), False


def save_checkpoint(
    pending_rows: list[dict[str, str]],
    results: list[dict[str, object]],
    failed_rows: list[dict[str, object]],
    runtime_state: dict[str, object],
) -> None:
    payload = {
        "pending_rows": pending_rows,
        "results": results,
        "failed_rows": failed_rows,
        "runtime_state": runtime_state,
    }
    CHECKPOINT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def clear_checkpoint() -> None:
    if CHECKPOINT_PATH.exists():
        CHECKPOINT_PATH.unlink()


def is_blocked_error(message: str | None) -> bool:
    if not message:
        return False
    markers = [
        "ERR_CONNECTION_CLOSED",
        "ERR_CONNECTION_RESET",
        "ERR_NETWORK_CHANGED",
        "ERR_HTTP2_PROTOCOL_ERROR",
    ]
    return any(marker in message for marker in markers)


def record_vpn_result(runtime_state: dict[str, object], switch_result: dict[str, object]) -> None:
    history = list(runtime_state.get("vpn_switch_history", []))
    history.append(switch_result)
    runtime_state["vpn_switch_history"] = history[-20:]
    runtime_state["last_vpn_result"] = switch_result
    runtime_state["vpn_target_index"] = switch_result.get("next_target_index", runtime_state.get("vpn_target_index", 0))
    if switch_result.get("current_ip"):
        runtime_state["last_public_ip"] = switch_result["current_ip"]
    target = switch_result.get("target")
    if target in VPN_FALLBACK_TARGETS:
        fallback_index = (VPN_FALLBACK_TARGETS.index(target) + 1) % len(VPN_FALLBACK_TARGETS)
        runtime_state["vpn_fallback_target_index"] = fallback_index


def build_vpn_target_sequence(runtime_state: dict[str, object]) -> list[str]:
    fallback_index = int(runtime_state.get("vpn_fallback_target_index", 0))
    rotated_fallback = VPN_FALLBACK_TARGETS[fallback_index:] + VPN_FALLBACK_TARGETS[:fallback_index]
    return [*VPN_PRIMARY_TARGETS, *rotated_fallback]


async def rotate_vpn_and_update_state(runtime_state: dict[str, object]) -> dict[str, object]:
    target_sequence = build_vpn_target_sequence(runtime_state)
    switch_result = await vpn_switcher.rotate_vpn_connection(
        targets=target_sequence,
        start_index=0,
        max_attempts=VPN_ROTATION_MAX_ATTEMPTS,
        connect_timeout_seconds=VPN_CONNECT_TIMEOUT_SECONDS,
        ready_timeout_seconds=VPN_READY_TIMEOUT_SECONDS,
        poll_interval_seconds=VPN_STATUS_POLL_INTERVAL_SECONDS,
        probe_target_url=VPN_PROBE_URL,
        probe_timeout_seconds=VPN_PROBE_TIMEOUT_SECONDS,
        max_probe_latency_seconds=VPN_MAX_PROBE_LATENCY_SECONDS,
    )
    switch_result["target_sequence"] = target_sequence
    record_vpn_result(runtime_state, switch_result)
    return switch_result


In [14]:
pending_rows, results, failed_rows, runtime_state, resumed_from_checkpoint = load_initial_targets()
current_vpn_status = await vpn_switcher.get_vpn_status()
runtime_state["last_public_ip"] = runtime_state.get("last_public_ip") or current_vpn_status.ip

print(f"체크포인트 재개 여부: {resumed_from_checkpoint}")
print(f"남은 대상 개수: {len(pending_rows)}")
print(f"기존 성공 개수: {len(results)}")
print(f"기존 실패 개수: {len(failed_rows)}")
print(json.dumps(current_vpn_status.to_dict(), ensure_ascii=False, indent=2))
print(json.dumps(runtime_state, ensure_ascii=False, indent=2))
pending_rows[:2]


체크포인트 재개 여부: True
남은 대상 개수: 7042
기존 성공 개수: 33
기존 실패 개수: 0
{
  "raw_output": "Status: Connected\nServer: South Korea #115\nHostname: kr115.nordvpn.com\nIP: 160.238.37.81\nCountry: South Korea\nCity: Seoul\nCurrent technology: NORDLYNX\nCurrent protocol: UDP\nPost-quantum VPN: Disabled\nTransfer: 300.46 MiB received, 30.34 MiB sent\nUptime: 23 minutes 56 seconds\n",
  "status": "Connected",
  "server": "South Korea #115",
  "hostname": "kr115.nordvpn.com",
  "ip": "160.238.37.81",
  "country": "South Korea",
  "city": "Seoul",
  "uptime": "23 minutes 56 seconds",
  "is_connected": true
}
{
  "vpn_target_index": 1,
  "vpn_fallback_target_index": 0,
  "vpn_switch_history": [
    {
      "success": true,
      "target": "Japan",
      "next_target_index": 1,
      "current_ip": "217.217.114.145",
      "error": null,
      "elapsed_seconds": 1.77,
      "attempted_targets": [
        "Japan"
      ],
      "status_before": {
        "raw_output": "Status: Disconnected\n",
        "status": 

[{'demo_no': '0361400836',
  'detail_url': 'https://www.carku.kr/search/car-detail.html?wDemoNo=0361400836',
  'source_page': '2',
  'source_index': 65},
 {'demo_no': '0361400850',
  'detail_url': 'https://www.carku.kr/search/car-detail.html?wDemoNo=0361400850',
  'source_page': '2',
  'source_index': 66}]

In [15]:
detail_semaphore = asyncio.Semaphore(DETAIL_CONCURRENCY)
pdf_semaphore = asyncio.Semaphore(PDF_CONCURRENCY)


async def process_one(row: dict[str, str]) -> dict[str, object]:
    detail_url = row["detail_url"]
    item_started_at = time.perf_counter()
    detail_page = None
    pdf_warning = None
    try:
        async with create_browser_context() as browser_context:
            async with detail_semaphore:
                detail_page = await crawl_detail_page(
                    detail_url,
                    timeout_ms=DETAIL_TIMEOUT_MS,
                    context=browser_context,
                )

            try:
                async with pdf_semaphore:
                    pdf_path = await download_performance_record_pdf(
                        detail_url,
                        OUTPUT_DIR,
                        timeout_ms=PDF_TIMEOUT_MS,
                        retry_count=PDF_RETRY_COUNT,
                        retry_backoff_seconds=PDF_RETRY_BACKOFF_SECONDS,
                        raise_on_blocked_error=True,
                        context=browser_context,
                    )
            except MissingPerformanceRecordError as exc:
                pdf_path = None
                pdf_warning = str(exc)

        item_elapsed = time.perf_counter() - item_started_at
        return {
            "status": "ok",
            "category": "ok",
            "order_index": row["source_index"],
            "detail_url": detail_url,
            "registration_number": detail_page.registration_number if detail_page else None,
            "detail_page": detail_page.to_dict() if detail_page else None,
            "performance_record_pdf": str(pdf_path) if pdf_path else None,
            "elapsed_seconds": round(item_elapsed, 2),
            "error": pdf_warning,
            "blocked": False,
        }
    except BlockedRequestError as exc:
        category = "blocked"
        error_message = str(exc)
    except SlowResponseError as exc:
        category = "slow"
        error_message = str(exc)
    except StaleListingError as exc:
        category = "stale"
        error_message = str(exc)
    except Exception as exc:  # noqa: BLE001
        error_message = str(exc)
        category = "blocked" if is_blocked_error(error_message) else "error"

    item_elapsed = time.perf_counter() - item_started_at
    return {
        "status": "error",
        "category": category,
        "order_index": row["source_index"],
        "detail_url": detail_url,
        "registration_number": None,
        "detail_page": detail_page.to_dict() if detail_page else None,
        "performance_record_pdf": None,
        "elapsed_seconds": round(item_elapsed, 2),
        "error": error_message,
        "blocked": category == "blocked",
    }


attempt_total = 0
total_started_at = time.perf_counter()
chunk_started_at = total_started_at

while pending_rows:
    current_batch = pending_rows[:BATCH_SIZE]
    remaining_rows = pending_rows[BATCH_SIZE:]
    tasks = [asyncio.create_task(process_one(row)) for row in current_batch]

    batch_results = []
    blocked_rows = []
    retry_later_rows = []
    slow_rows = []

    for task in asyncio.as_completed(tasks):
        result = await task
        batch_results.append(result)
        attempt_total += 1
        print(
            f"[시도 {attempt_total}] 완료 - 차량번호: {result.get('registration_number')} / PDF: {result.get('performance_record_pdf')} / 소요: {result.get('elapsed_seconds')}초 / 오류: {result.get('error')}"
        )

        if attempt_total % LOG_EVERY == 0:
            chunk_elapsed = time.perf_counter() - chunk_started_at
            print(f"[로그] 최근 {LOG_EVERY}번 시도 시간: {chunk_elapsed:.2f}초")
            chunk_started_at = time.perf_counter()

    for row, result in zip(current_batch, sorted(batch_results, key=lambda item: item['order_index'])):
        if result['status'] == 'ok':
            results.append(result)
            continue

        if result['category'] == 'blocked':
            blocked_rows.append(row)
            continue

        if result['category'] == 'slow':
            slow_retry_count = int(row.get('slow_retry_count', 0)) + 1
            slow_row = {**row, 'slow_retry_count': slow_retry_count}
            if slow_retry_count >= SLOW_TO_BLOCK_THRESHOLD:
                blocked_rows.append(slow_row)
            elif slow_retry_count <= SLOW_RETRY_LIMIT:
                retry_later_rows.append(slow_row)
                slow_rows.append(slow_row)
            else:
                failed_rows.append({**result, 'slow_retry_count': slow_retry_count})
            continue

        failed_rows.append(result)

    if len(blocked_rows) == 0 and len(slow_rows) >= BATCH_SLOW_BLOCK_THRESHOLD:
        blocked_rows = slow_rows
        retry_later_rows = []
        print(f"느린 응답이 배치에서 {len(slow_rows)}건 발생하여 차단 의심으로 승격합니다.")

    pending_rows = blocked_rows + remaining_rows + retry_later_rows
    save_checkpoint(pending_rows, results, failed_rows, runtime_state)

    if len(blocked_rows) >= BLOCKED_ERROR_THRESHOLD:
        print("차단 의심 오류 감지. 체크포인트 저장 후 VPN 자동 전환을 시도합니다.")
        switch_result = await rotate_vpn_and_update_state(runtime_state)
        save_checkpoint(pending_rows, results, failed_rows, runtime_state)
        print(json.dumps(switch_result, ensure_ascii=False, indent=2))

        if not switch_result.get("success"):
            print("VPN 전환 실패. 체크포인트를 유지한 채 여기서 중단합니다.")
            break

        print("VPN 전환 완료. 같은 pending rows로 다시 진행합니다.")
        chunk_started_at = time.perf_counter()
        continue

if not pending_rows:
    clear_checkpoint()

if attempt_total % LOG_EVERY != 0:
    chunk_size = attempt_total % LOG_EVERY
    chunk_elapsed = time.perf_counter() - chunk_started_at
    print(f"[로그] 최근 {chunk_size}번 시도 시간: {chunk_elapsed:.2f}초")

total_elapsed = time.perf_counter() - total_started_at
print(f"전체 소요 시간: {total_elapsed:.2f}초")
print(f"남은 대상 개수: {len(pending_rows)}")
print(f"성공 개수: {len(results)}")
print(f"실패 개수: {len(failed_rows)}")


[시도 1] 완료 - 차량번호: None / PDF: None / 소요: 20.57초 / 오류: detail page timeout
[시도 2] 완료 - 차량번호: None / PDF: None / 소요: 20.58초 / 오류: detail page timeout
[시도 3] 완료 - 차량번호: None / PDF: None / 소요: 20.59초 / 오류: detail page timeout
느린 응답이 배치에서 3건 발생하여 차단 의심으로 승격합니다.
차단 의심 오류 감지. 체크포인트 저장 후 VPN 자동 전환을 시도합니다.
{
  "success": true,
  "target": "kr106",
  "next_target_index": 3,
  "current_ip": "89.147.101.126",
  "error": null,
  "elapsed_seconds": 1.63,
  "attempted_targets": [
    "kr115",
    "kr142",
    "kr106"
  ],
  "status_before": {
    "raw_output": "Status: Disconnected\n",
    "status": "Disconnected",
    "server": null,
    "hostname": null,
    "ip": null,
    "country": null,
    "city": null,
    "uptime": null,
    "is_connected": false
  },
  "status_after": {
    "raw_output": "Status: Connected\nServer: South Korea #106\nHostname: kr106.nordvpn.com\nIP: 89.147.101.126\nCountry: South Korea\nCity: Seoul\nCurrent technology: NORDLYNX\nCurrent protocol: UDP\nPost-quantum VPN: Disab

CancelledError: 

In [ ]:
print(json.dumps(results, ensure_ascii=False, indent=2))
print(json.dumps(failed_rows, ensure_ascii=False, indent=2))
print(json.dumps(runtime_state, ensure_ascii=False, indent=2))
print(pending_rows)
